# LT-DAC vs E-DAC: direct and hidden cost notes

This notebook compares low-temperature DAC (LT-DAC) and electricity-driven DAC (E-DAC) on a GBP/tCO2 basis and then tests system-level combinations.

The direct comparison asks which technology is cheaper per Mt of gross removal using the assumptions in `src.assumptions`.

The hidden-cost comparison is more important:

- LT-DAC is only available when gas CCS generation runs, because the model uses gas waste heat as the heat source.
- Gas generation creates CO2 emissions, so LT-DAC gross capture must be netted against gas emissions.
- E-DAC only runs on surplus electricity after medium storage and hydrogen storage have been filled.
- A large E-DAC capacity can still remove little CO2 if the system has too little surplus.

This notebook uses the with-H2 `PowerSystem` model, not the `PowerSystemNoH2` model.

In [ ]:
from __future__ import annotations

import itertools

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import src.assumptions as A
import src.matplotlib_style  # noqa: F401
from src.data import renewable_capacity_factors
from src.demand_model import DemandMode, predicted_demand
from src.power_system import PowerSystem
from src.supply_model import daily_renewables_capacity
from src.units import Units as U

## 1. Direct cost per tonne

This first table is the simple, technology-only comparison. It does not check whether the system actually has surplus electricity for E-DAC or gas waste heat for LT-DAC.

For E-DAC there are two useful views:

- `assumption_lcor_gbp_per_tco2`: the direct LCOR stored in assumptions.
- `electricity_only_gbp_per_tco2`: the model's electricity cost implied by DAC energy demand and `A.EDAC.LCOE`.

For LT-DAC, the direct cost is `A.LTDAC.LCOR`.

In [ ]:
def gbp_per_t(cost: object) -> float:
    return cost.to(U.GBP / U.t).magnitude


edac_energy_twh_per_mt = A.DAC.EnergyCost.MediumTWhPerMtCO2.to(U.TWh / U.Mt)
edac_electricity_cost_per_mt = (edac_energy_twh_per_mt * A.EDAC.LCOE).to(U.GBP / U.Mt)
edac_electricity_cost_per_t = edac_electricity_cost_per_mt.to(U.GBP / U.t)

direct_costs = pd.DataFrame(
    [
        {
            "technology": "E-DAC",
            "assumption_lcor_gbp_per_tco2": gbp_per_t(A.EDAC.LCOR),
            "electricity_only_gbp_per_tco2": gbp_per_t(edac_electricity_cost_per_t),
            "energy_twh_per_mtco2": edac_energy_twh_per_mt.magnitude,
        },
        {
            "technology": "LT-DAC",
            "assumption_lcor_gbp_per_tco2": gbp_per_t(A.LTDAC.LCOR),
            "electricity_only_gbp_per_tco2": np.nan,
            "energy_twh_per_mtco2": A.LTDAC.GasInputPerTonCO2.to(U.TWh / U.Mt).magnitude,
        },
    ]
)

display(direct_costs.round(3))

cheaper_direct = direct_costs.sort_values("assumption_lcor_gbp_per_tco2").iloc[0]
print(
    "Direct LCOR winner: "
    f"{cheaper_direct['technology']} at {cheaper_direct['assumption_lcor_gbp_per_tco2']:.2f} GBP/tCO2"
)

## 2. Scenario design for hidden costs

The grid below changes renewable capacity, E-DAC capacity, gas CCS capacity, and daily demand. Hydrogen storage and electrolysis are left at the default assumptions so this remains a with-H2 test.

`DEMAND_MULTIPLIERS` scales every day in the demand series. For example, `1.10` is a 10% higher daily demand system. This is a compact way to test which DAC mix is cheaper under easier or harder demand conditions.

The default grid is intentionally moderate. Increase the ranges or reduce the step size for a final search.

In [ ]:
TARGET_NET_CAPTURE_MT = 35.8

CAPACITY_FACTORS_SOURCE = "era5_2024"
ENABLE_IMPORTS = False

DEMAND_MODE_GRID = [DemandMode.SEASONAL]
DEMAND_MULTIPLIERS = [0.90, 1.00, 1.10]

RENEWABLE_CAPACITY_GRID_GW = np.arange(240, 501, 40)
E_DAC_CAPACITY_GRID_GW = np.array([0, 1, 2, 3, 4, 5, 6, 8], dtype=float)
GAS_CAPACITY_GRID_GW = np.array([0, 5, 10, 15, 20, 25, 30, 40, 50, 60, 80], dtype=float)

ONLY_DAC_IF_HYDROGEN_STORAGE_FULL = True
ENABLE_GAS_LTDAC = True

In [ ]:
daily_capacity_factors = renewable_capacity_factors.get_renewable_capacity_factors(
    source=CAPACITY_FACTORS_SOURCE,
    resample="D",
)


def build_demand(mode: DemandMode, demand_multiplier: float) -> pd.DataFrame:
    demand = predicted_demand(mode=mode, historical="era5", filter_ldz=True).copy()
    demand["demand"] = demand["demand"] * demand_multiplier
    return demand


def capacity_label(renewable_capacity_gw: float) -> int | float:
    return int(renewable_capacity_gw) if float(renewable_capacity_gw).is_integer() else renewable_capacity_gw


def build_net_supply_df(renewable_capacity_gw: float, demand_df: pd.DataFrame) -> pd.DataFrame:
    renewable_capacity_gw = capacity_label(renewable_capacity_gw)
    supply = daily_renewables_capacity(renewable_capacity_gw * U.GW, daily_capacity_factors)
    supply = supply * (1 - A.PowerSystem.TotalLosses)
    common_idx = supply.index.intersection(demand_df.index)
    assert len(common_idx) > 0, "No common dates between supply and demand."

    net_supply = supply.reindex(common_idx) - demand_df.reindex(common_idx)["demand"]
    return pd.DataFrame({renewable_capacity_gw: net_supply})


demand_by_case = {
    (mode.value, demand_multiplier): build_demand(mode, demand_multiplier)
    for mode, demand_multiplier in itertools.product(DEMAND_MODE_GRID, DEMAND_MULTIPLIERS)
}

print(f"Prepared {len(demand_by_case)} demand cases.")

## 3. Run one scenario

`annual_net_capture_mt` is the main feasibility metric:

`E-DAC gross capture + LT-DAC gross capture - gas emissions`

The all-in cost includes the current power-system cost calculation plus the DAC costs reported by the model. This is the metric used to choose the cheapest feasible combination.

In [ ]:
def run_system_case(
    *,
    demand_mode: DemandMode,
    demand_multiplier: float,
    renewable_capacity_gw: float,
    e_dac_capacity_gw: float,
    gas_capacity_gw: float,
) -> dict[str, float | str | bool]:
    renewable_capacity_gw = capacity_label(renewable_capacity_gw)
    demand_df = demand_by_case[demand_mode.value, demand_multiplier]
    net_supply_df = build_net_supply_df(renewable_capacity_gw, demand_df)

    system = PowerSystem(
        renewable_capacity=renewable_capacity_gw * U.GW,
        hydrogen_storage_capacity=A.HydrogenStorage.CavernStorage.Capacity,
        electrolyser_power=A.HydrogenStorage.Electrolysis.Power,
        hydrogen_generation_power=A.HydrogenStorage.Generation.Power,
        dac_capacity=e_dac_capacity_gw * U.GW,
        gas_ccs_capacity=gas_capacity_gw * U.GW,
        only_dac_if_hydrogen_storage_full=ONLY_DAC_IF_HYDROGEN_STORAGE_FULL,
        enable_gas_ltdac=ENABLE_GAS_LTDAC,
        enable_imports=ENABLE_IMPORTS,
        capacity_factors_source=CAPACITY_FACTORS_SOURCE,
    )

    sim_df = system.run_simulation(net_supply_df)
    analysis = system.analyze_simulation_results(sim_df)
    if sim_df is None or analysis is None:
        return {
            "status": "simulation_failed",
            "demand_mode": demand_mode.value,
            "demand_multiplier": demand_multiplier,
            "renewable_capacity_gw": renewable_capacity_gw,
            "e_dac_capacity_gw": e_dac_capacity_gw,
            "gas_capacity_gw": gas_capacity_gw,
            "target_feasible": False,
        }

    annual_power_system_cost = system.calculate_power_system_cost(sim_df).to(U.GBP).magnitude
    annual_dac_cost = analysis["annual_dac_total_cost"].to(U.GBP).magnitude
    total_cost = annual_power_system_cost + annual_dac_cost

    annual_net_capture_mt = analysis["annual_net_capture"].to(U.Mt).magnitude
    annual_grid_dac_capture_mt = analysis["annual_co2_removals"].to(U.Mt).magnitude
    annual_ltdac_capture_mt = analysis["annual_gas_dac_capture"].to(U.Mt).magnitude
    annual_gas_emissions_mt = analysis["annual_gas_emissions"].to(U.Mt).magnitude
    annual_gas_twh = analysis["annual_gas_ccs_energy"].to(U.TWh).magnitude
    annual_edac_energy_twh = analysis["annual_dac_energy"].to(U.TWh).magnitude
    annual_ltdac_electricity_twh = analysis["annual_gas_dac_electricity"].to(U.TWh).magnitude
    curtailed_twh = analysis["curtailed_energy"].to(U.TWh).magnitude

    edac_max_energy_twh = (e_dac_capacity_gw * U.GW * A.HoursPerYear).to(U.TWh).magnitude
    edac_energy_utilization = annual_edac_energy_twh / edac_max_energy_twh if edac_max_energy_twh > 0 else 0.0

    return {
        "status": "ok",
        "demand_mode": demand_mode.value,
        "demand_multiplier": demand_multiplier,
        "renewable_capacity_gw": renewable_capacity_gw,
        "e_dac_capacity_gw": e_dac_capacity_gw,
        "gas_capacity_gw": gas_capacity_gw,
        "target_feasible": annual_net_capture_mt >= TARGET_NET_CAPTURE_MT,
        "annual_net_capture_mt": annual_net_capture_mt,
        "annual_grid_dac_capture_mt": annual_grid_dac_capture_mt,
        "annual_ltdac_capture_mt": annual_ltdac_capture_mt,
        "annual_gas_emissions_mt": annual_gas_emissions_mt,
        "annual_gas_twh": annual_gas_twh,
        "annual_edac_energy_twh": annual_edac_energy_twh,
        "annual_ltdac_electricity_twh": annual_ltdac_electricity_twh,
        "curtailed_twh": curtailed_twh,
        "edac_energy_utilization": edac_energy_utilization,
        "annual_power_system_cost_gbp": annual_power_system_cost,
        "annual_dac_cost_gbp": annual_dac_cost,
        "total_cost_gbp_yr": total_cost,
        "total_cost_billion_gbp_yr": total_cost / 1e9,
        "total_cost_gbp_per_mwh": (total_cost * U.GBP / A.EnergyDemand2050).to(U.GBP / U.MWh).magnitude,
        "dac_cost_gbp_per_net_tco2": annual_dac_cost / (annual_net_capture_mt * 1e6) if annual_net_capture_mt > 0 else np.nan,
        "system_cost_gbp_per_net_tco2": total_cost / (annual_net_capture_mt * 1e6) if annual_net_capture_mt > 0 else np.nan,
    }

## 4. Grid search

The grid search keeps failed simulations so it is clear where the system cannot meet demand. Feasible rows meet the target annual net removal.

In [ ]:
rows: list[dict[str, float | str | bool]] = []

grid = list(
    itertools.product(
        DEMAND_MODE_GRID,
        DEMAND_MULTIPLIERS,
        RENEWABLE_CAPACITY_GRID_GW,
        E_DAC_CAPACITY_GRID_GW,
        GAS_CAPACITY_GRID_GW,
    )
)

for i, (demand_mode, demand_multiplier, renewable_gw, e_dac_gw, gas_gw) in enumerate(grid, start=1):
    rows.append(
        run_system_case(
            demand_mode=demand_mode,
            demand_multiplier=float(demand_multiplier),
            renewable_capacity_gw=float(renewable_gw),
            e_dac_capacity_gw=float(e_dac_gw),
            gas_capacity_gw=float(gas_gw),
        )
    )
    if i % 100 == 0 or i == len(grid):
        print(f"Finished {i}/{len(grid)} cases")

results = pd.DataFrame(rows)
ok_results = results[results["status"] == "ok"].copy()
feasible = ok_results[ok_results["target_feasible"]].copy()

print(f"Total cases: {len(results):,}")
print(f"Successful simulations: {len(ok_results):,}")
print(f"Feasible target cases: {len(feasible):,}")

## 5. Cheapest feasible combinations

The cheapest row can be a mix of E-DAC and LT-DAC. Use the capture columns to see which mechanism is doing the work.

In [ ]:
summary_cols = [
    "demand_mode",
    "demand_multiplier",
    "renewable_capacity_gw",
    "e_dac_capacity_gw",
    "gas_capacity_gw",
    "annual_net_capture_mt",
    "annual_grid_dac_capture_mt",
    "annual_ltdac_capture_mt",
    "annual_gas_emissions_mt",
    "annual_gas_twh",
    "curtailed_twh",
    "edac_energy_utilization",
    "total_cost_billion_gbp_yr",
    "total_cost_gbp_per_mwh",
    "dac_cost_gbp_per_net_tco2",
    "system_cost_gbp_per_net_tco2",
]

if feasible.empty:
    print("No feasible combination found. Increase DAC capacity, gas capacity, renewable capacity, or relax the target.")
else:
    best = feasible.sort_values("total_cost_gbp_yr").iloc[0]
    display(feasible.sort_values("total_cost_gbp_yr")[summary_cols].head(20).round(3))

    print(
        "Cheapest feasible combination:\n"
        f"- Demand mode: {best['demand_mode']}\n"
        f"- Demand multiplier: {best['demand_multiplier']:.2f}\n"
        f"- Renewables: {best['renewable_capacity_gw']:.0f} GW\n"
        f"- E-DAC: {best['e_dac_capacity_gw']:.1f} GW\n"
        f"- Gas CCS: {best['gas_capacity_gw']:.1f} GW\n"
        f"- Net capture: {best['annual_net_capture_mt']:.2f} MtCO2/yr\n"
        f"- E-DAC gross capture: {best['annual_grid_dac_capture_mt']:.2f} MtCO2/yr\n"
        f"- LT-DAC gross capture: {best['annual_ltdac_capture_mt']:.2f} MtCO2/yr\n"
        f"- Gas emissions: {best['annual_gas_emissions_mt']:.2f} MtCO2/yr\n"
        f"- Total cost: {best['total_cost_billion_gbp_yr']:.2f} bn GBP/yr\n"
        f"- System cost per net tCO2: {best['system_cost_gbp_per_net_tco2']:.2f} GBP/tCO2"
    )

## 6. Cheapest combination under each daily demand level

This table answers the demand-sensitivity question directly. If E-DAC utilization collapses at high demand, the system has too little surplus for that E-DAC capacity. If LT-DAC capture is high, the system is relying on gas operation and should be checked against gas emissions.

In [ ]:
if feasible.empty:
    print("No feasible combinations to summarize.")
else:
    best_by_demand = (
        feasible.sort_values(["demand_mode", "demand_multiplier", "total_cost_gbp_yr"])
        .groupby(["demand_mode", "demand_multiplier"], as_index=False)
        .first()
    )
    display(best_by_demand[summary_cols].round(3))

    fig, ax = plt.subplots(figsize=(8, 5))
    scatter = ax.scatter(
        best_by_demand["demand_multiplier"],
        best_by_demand["total_cost_billion_gbp_yr"],
        c=best_by_demand["e_dac_capacity_gw"],
        s=40 + best_by_demand["gas_capacity_gw"] * 6,
        cmap="viridis",
        alpha=0.85,
        edgecolor="black",
        linewidth=0.5,
    )
    for _, row in best_by_demand.iterrows():
        ax.annotate(
            f"R{row['renewable_capacity_gw']:.0f} E{row['e_dac_capacity_gw']:.1f} G{row['gas_capacity_gw']:.0f}",
            (row["demand_multiplier"], row["total_cost_billion_gbp_yr"]),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=8,
        )
    ax.set_xlabel("Daily demand multiplier")
    ax.set_ylabel("Total annual cost (bn GBP/yr)")
    ax.set_title("Cheapest feasible mix by daily demand multiplier")
    ax.grid(True, alpha=0.25)
    plt.colorbar(scatter, ax=ax, label="E-DAC capacity (GW)")
    plt.show()

## 7. E-DAC surplus constraint and LT-DAC gas constraint

The next plots separate the two hidden constraints:

- E-DAC utilization measures how much of installed E-DAC capacity actually runs.
- LT-DAC capture and gas emissions show whether waste-heat capture is coming with a large gas penalty.

In [ ]:
if ok_results.empty:
    print("No successful simulations to plot.")
else:
    plot_df = ok_results.copy()
    plot_df["net_capture_shortfall_mt"] = TARGET_NET_CAPTURE_MT - plot_df["annual_net_capture_mt"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].scatter(
        plot_df["curtailed_twh"],
        plot_df["edac_energy_utilization"],
        c=plot_df["demand_multiplier"],
        cmap="viridis",
        s=24,
        alpha=0.7,
    )
    axes[0].set_xlabel("Annual curtailed energy (TWh)")
    axes[0].set_ylabel("E-DAC energy utilization")
    axes[0].set_title("E-DAC depends on surplus")
    axes[0].grid(True, alpha=0.25)

    scatter = axes[1].scatter(
        plot_df["annual_gas_emissions_mt"],
        plot_df["annual_ltdac_capture_mt"],
        c=plot_df["annual_gas_twh"],
        cmap="plasma",
        s=24,
        alpha=0.7,
    )
    axes[1].plot([0, plot_df["annual_gas_emissions_mt"].max()], [0, plot_df["annual_gas_emissions_mt"].max()], "k--", linewidth=1)
    axes[1].set_xlabel("Annual gas emissions (MtCO2)")
    axes[1].set_ylabel("Annual LT-DAC gross capture (MtCO2)")
    axes[1].set_title("LT-DAC depends on gas operation")
    axes[1].grid(True, alpha=0.25)
    plt.colorbar(scatter, ax=axes[1], label="Annual gas generation (TWh)")

    plt.tight_layout()
    plt.show()

## 8. Optional finer search around the current winner

Set `RUN_FINE_SEARCH = True` after the broad grid finds a plausible winner. This searches a smaller region with finer capacity steps.

In [ ]:
RUN_FINE_SEARCH = False

if RUN_FINE_SEARCH and not feasible.empty:
    best = feasible.sort_values("total_cost_gbp_yr").iloc[0]
    fine_renewables = np.arange(max(100, best["renewable_capacity_gw"] - 30), best["renewable_capacity_gw"] + 31, 10)
    fine_edac = np.round(np.arange(max(0, best["e_dac_capacity_gw"] - 1.0), best["e_dac_capacity_gw"] + 1.01, 0.25), 3)
    fine_gas = np.round(np.arange(max(0, best["gas_capacity_gw"] - 10), best["gas_capacity_gw"] + 10.01, 2.5), 3)

    fine_rows = []
    for renewable_gw, e_dac_gw, gas_gw in itertools.product(fine_renewables, fine_edac, fine_gas):
        fine_rows.append(
            run_system_case(
                demand_mode=DemandMode(best["demand_mode"]),
                demand_multiplier=float(best["demand_multiplier"]),
                renewable_capacity_gw=float(renewable_gw),
                e_dac_capacity_gw=float(e_dac_gw),
                gas_capacity_gw=float(gas_gw),
            )
        )

    fine_results = pd.DataFrame(fine_rows)
    fine_feasible = fine_results[(fine_results["status"] == "ok") & fine_results["target_feasible"]].copy()
    display(fine_feasible.sort_values("total_cost_gbp_yr")[summary_cols].head(20).round(3))
else:
    print("Fine search skipped.")